In [ ]:
# import torch
# import numpy as np
# from datasets import load_dataset
# import evaluate
# from transformers import (
#     TrainingArguments,
#     Trainer,
#     DataCollatorWithPadding,
#     GPT2Tokenizer
# )

# from gpt_classifier import GPTForSequenceClassification
# from load_model import load_pretrained_model


In [ ]:
# # Choose variant: "ipa", "normal", or "prebuilt"
# VARIANT = "ipa"

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# CHECKPOINT_PATHS = {
#     "ipa": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt",
#     "normal": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt",
#     "prebuilt": "/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_medium/ckpt.pt"
# }

# TOKENIZER_PATHS = {
#     "ipa": "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation",
#     "normal": "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation"
# }


In [ ]:
# if VARIANT == "prebuilt":
#     tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
#     tokenizer.pad_token = tokenizer.eos_token
# else:
#     from tokenizer import load_tokenizer
#     vocab_path = TOKENIZER_PATHS[VARIANT] + "-vocab.json"
#     merges_path = TOKENIZER_PATHS[VARIANT] + "-merges.txt"
#     tokenizer = load_tokenizer(vocab_path, merges_path)

In [ ]:
# base_model = load_pretrained_model(CHECKPOINT_PATHS[VARIANT], device=DEVICE)
# base_model.config.pad_token_id = tokenizer.pad_token_id
# model = GPTForSequenceClassification(base_model, num_labels=2).to(DEVICE)


In [ ]:
# dataset = load_dataset("iggy12345/glue-rte-ipa")
# train_dataset = dataset["train"]
# eval_dataset = dataset["validation"]

# def preprocess(example):
#     if VARIANT in ["ipa", "normal"]:
#         text = example["sentence1-phoneme"] + " ? " + example["sentence2-phoneme"]
#     else:
#         text = example["sentence1"] + " ? " + example["sentence2"]
#     return tokenizer(text, truncation=True, padding="max_length", max_length=128)

# encoded_train = train_dataset.map(preprocess)
# encoded_eval = eval_dataset.map(preprocess)


In [ ]:
# accuracy = evaluate.load("accuracy")

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     preds = np.argmax(logits, axis=-1)
#     return accuracy.compute(predictions=preds, references=labels)

# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# training_args = TrainingArguments(
#     output_dir=f"./rte-{VARIANT}",
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-5,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     num_train_epochs=8,
#     weight_decay=0.01,
#     logging_dir=f"./logs-{VARIANT}",
#     report_to="none",
#     fp16=torch.cuda.is_available(),
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=encoded_train,
#     eval_dataset=encoded_eval,
#     tokenizer=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics
# )


In [ ]:
# trainer.train()


In [1]:
import torch
import numpy as np
from datasets import load_dataset
import evaluate
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    GPT2Tokenizer
)

from gpt_classifier import GPTForSequenceClassification
from load_model import load_pretrained_model


/users/PAS2836/krishnakb/ondemand/krishna_proj/cleanenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Paths to checkpoints
CHECKPOINT_PATHS = {
    "ipa": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt",
    "normal": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt",
#     "prebuilt": "/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_medium/ckpt.pt"
}
# CHECKPOINT_PATHS = {
#     "ipa": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_50k/ckpt.pt",
#     "normal": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_50k/ckpt.pt",
#     "prebuilt": "/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_50k/ckpt.pt"  # 👈 update path if needed
# }

# Tokenizer base paths (no .json/.txt yet)
TOKENIZER_PATHS = {
    "ipa": "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation",
    "normal": "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation"
}

# Load RTE phoneme dataset once
dataset = load_dataset("iggy12345/glue-rte-ipa")
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]


In [3]:
variants = ["ipa", "normal", "prebuilt"]
results = []

for VARIANT in variants:
    print(f"\n🚀 Starting training for VARIANT = {VARIANT.upper()}")

    # Load tokenizer
    if VARIANT == "prebuilt":
        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token
    else:
        from tokenizer import load_tokenizer
        vocab_path = TOKENIZER_PATHS[VARIANT] + "-vocab.json"
        merges_path = TOKENIZER_PATHS[VARIANT] + "-merges.txt"
        tokenizer = load_tokenizer(vocab_path, merges_path)

    # Load model
    try:
        base_model = load_pretrained_model(CHECKPOINT_PATHS[VARIANT], device=DEVICE)
    except Exception as e:
        print(f"⚠️ Failed to load checkpoint for {VARIANT}: {e}")
        continue

    base_model.config.pad_token_id = tokenizer.pad_token_id
    model = GPTForSequenceClassification(base_model, num_labels=2).to(DEVICE)

    # Preprocessing
    def preprocess(example):
        if VARIANT in ["ipa", "normal"]:
            text = f"Premise: {example['sentence1-phoneme']} Hypothesis: {example['sentence2-phoneme']}"
        else:
            text = f"Premise: {example['sentence1']} Hypothesis: {example['sentence2']}"
    
        return tokenizer(text, truncation=True, padding="max_length", max_length=128)

    encoded_train = train_dataset.map(preprocess)
    encoded_eval = eval_dataset.map(preprocess)

    # Oversample minority class (label == 1)
    from datasets import concatenate_datasets

    minority = encoded_train.filter(lambda x: x["label"] == 1)
    oversampled_train = concatenate_datasets([encoded_train, minority])
    oversampled_train = oversampled_train.shuffle(seed=42)

    # Metric
    accuracy = evaluate.load("accuracy")
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return accuracy.compute(predictions=preds, references=labels)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = TrainingArguments(
        output_dir=f"./rte-small-{VARIANT}",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-5,
        warmup_ratio=0.2,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.01,
        logging_dir=f"./logs-small-{VARIANT}",
        report_to="none",
        fp16=torch.cuda.is_available(),
        gradient_accumulation_steps=1,
        lr_scheduler_type="linear",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=oversampled_train,  # <-- Use oversampled_train here
        eval_dataset=encoded_eval,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # Train
    trainer.train()

    # Evaluate + store metrics
    eval_metrics = trainer.evaluate()
    train_loss = trainer.state.log_history[-1].get("loss", None)
    eval_loss = eval_metrics.get("eval_loss", None)
    accuracy_score = eval_metrics.get("eval_accuracy", None)

    results.append({
        "Variant": VARIANT,
        "Train Loss": round(train_loss, 4) if train_loss else None,
        "Eval Loss": round(eval_loss, 4) if eval_loss else None,
        "Eval Accuracy (%)": round(accuracy_score * 100, 2) if accuracy_score else None
    })



🚀 Starting training for VARIANT = IPA
number of parameters: 353.24M


/users/PAS2836/krishnakb/ondemand/krishna_proj/cleanenv/lib/python3.12/site-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/slurmtmp.1371878/ipykernel_1586631/2063040514.py:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.759269,0.472924
2,No log,0.883370,0.472924
3,0.693400,0.775666,0.472924
4,0.693400,0.761024,0.472924


OSError: [Errno 122] Disk quota exceeded

In [ ]:
import pandas as pd

# Sort and reset index
df = pd.DataFrame(results)
df = df.sort_values(by="Variant").reset_index(drop=True)

# Display the table
df.head()  # Shows the top rows of the table in Jupyter


In [ ]:
# torch.load("/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_medium/ckpt.pt")

In [4]:
!nvidia-smi


Tue Jun 24 13:00:54 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.12              Driver Version: 550.90.12      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:25:00.0 Off |                    0 |
| N/A   40C    P0             41W /  250W |   19999MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----